# Guía L2 — El índice sancocho · **Sesión 1: la receta como función**

**Cálculo Diferencial — Laboratorio computacional** · Universidad Nacional de Colombia, Sede Manizales

> Esta idea nació un domingo en Berlín, entre un döner y el recuerdo de una bullabesa. La historia completa está en la guía: este cuaderno es la sobremesa vuelta código.

**Uso en Colab:** guarden su copia (`Archivo → Guardar una copia en Drive`) y renómbrenla `L2_apellidos.ipynb`. Las celdas marcadas con ✏️ invitan a explorar o modificar; las marcadas con 🗣️ son para escribir conclusiones del equipo.

---
## ⏸️ Momento 1 — Predecir

Antes de ejecutar, respondan P1, P2 y P3 de la guía en papel: el costo del sancocho para seis en Manizales, si manda el espinazo o el resto de la olla, y si subió más la papa o el plátano. Foto al informe.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Precios mayoristas mensuales de SIPSA-DANE, congelados por el curso.
# El archivo .csv.gz se lee directamente: pandas reconoce la compresión.
RUTA = (
    "https://raw.githubusercontent.com/cmorregof/labs-calculo-2026-2/"
    "main/datos/sipsa_mensual.csv.gz"
)
df = pd.read_csv(RUTA)

print("Filas y columnas:", df.shape)
df.head(8)

---
## 1 · El ritual, ya sin niñera

En L1 lo hicimos juntos; hoy es de ustedes. El objetivo es decidir qué valores acepta cada columna, buscar valores ilegales y entender el alcance del congelado. Después, busquen los nombres oficiales de los ingredientes: los datos no saben cómo los llamamos en la casa.

🗣️ Escriban como comentario sus decisiones sobre los valores faltantes, precios no positivos y filas repetidas. ¿Una fila por `(fecha, mercado, producto)` es lo esperado?

In [ ]:
# ✏️ Ejercicio 1 — Completen la auditoría y dejen sus decisiones en comentarios.
print("Productos distintos:", df["producto"].nunique())
print("Ciudades:", df["ciudad"].nunique())
print("Meses: de", df["fecha"].min(), "a", df["fecha"].max())
print("\nTipos de dato:\n", df.dtypes)
print("\nValores faltantes por columna:\n", df.isna().sum())
print("\nPrecios medios no positivos:", (df["precio_kg"] <= 0).sum())
print("Filas repetidas por fecha, mercado y producto:",
      df.duplicated(["fecha", "mercado", "producto"]).sum())

# ¿Qué dominio matemático le asignan a fecha, ciudad, mercado, producto y precio_kg?

In [ ]:
# ✏️ Ejercicio 2 — Cambien `busca` e iteren.
# Encuentren los nombres oficiales de espinazo, yuca, papa, mazorca y plátano verde.
# Luego restrinjan la búsqueda a Manizales: no todas las variedades se venden allí.
busca = "espinazo"
coincidencias = df.loc[
    df["producto"].str.contains(busca, case=False, na=False),
    ["ciudad", "mercado", "producto"],
]
print(coincidencias["producto"].drop_duplicates().to_list())

# Pista para Manizales:
# df.loc[(df["ciudad"] == "Manizales") &
#        df["producto"].str.contains(busca, case=False, na=False), "producto"].unique()

---
## 2 · La receta en código

La receta oficial es para seis personas. Cada valor está en kilogramos, que es la unidad del mercado: 1,5 kg de espinazo, 1 kg de yuca, 1 kg de papa, 0,9 kg de mazorca y 1,4 kg de plátano. Traducir de unidades de cocina a kilogramos también es una función.

In [ ]:
RECETA = {
    "Carne de cerdo, espinazo":       1.5,
    "Yuca ICA":                       1.0,
    "Papa capira":                    1.0,
    "Chócolo mazorca":                0.9,
    "Plátano dominico hartón verde":  1.4,
}
print("Ingredientes:", len(RECETA), "| Kilos totales:", sum(RECETA.values()))

# ✏️ Confirmen que cada nombre existe para Manizales antes de seguir.

---
## 3 · El pipeline, pieza por pieza

Estas cuatro funciones son puras: reciben una tabla y sus parámetros, y devuelven una tabla o un número. No dependen de `df`, `RECETA` ni ninguna otra variable externa. Lean cada contrato, prueben cada función por separado y luego sigan el dato a través de la composición.

In [ ]:
def filtrar_ciudad(tabla, ciudad):
    """tabla → tabla: conserva únicamente las filas de una ciudad."""
    return tabla.loc[tabla["ciudad"] == ciudad].copy()

def filtrar_mes(tabla, fecha):
    """tabla → tabla: conserva únicamente el mes AAAA-MM indicado."""
    return tabla.loc[tabla["fecha"] == fecha].copy()

def filtrar_productos(tabla, receta):
    """tabla → tabla: conserva los productos que aparecen en la receta."""
    return tabla.loc[tabla["producto"].isin(receta)].copy()

def costear(tabla, receta):
    """tabla → número: suma kilos × precio_kg de cada ingrediente."""
    precios = tabla.groupby("producto")["precio_kg"].mean()
    faltantes = [producto for producto in receta if producto not in precios]
    if faltantes:
        raise ValueError(f"Faltan ingredientes: {faltantes}")
    return sum(kilos * precios[producto] for producto, kilos in receta.items())

# ✏️ ¿Por qué `.isin(receta)` sirve para varios productos y `== receta` no?
# ¿En qué situación habría varias filas del mismo ingrediente y por qué usamos `mean`?

In [ ]:
# Una función que fabrica la función compuesta.
def componer(*funciones):
    def compuesta(x):
        for funcion in funciones:
            x = funcion(x)
        return x
    return compuesta

CIUDAD = "Manizales"
MES = df.loc[df["ciudad"] == CIUDAD, "fecha"].max()

costo_sancocho = componer(
    lambda tabla: filtrar_ciudad(tabla, CIUDAD),
    lambda tabla: filtrar_mes(tabla, MES),
    lambda tabla: filtrar_productos(tabla, RECETA),
    lambda tabla: costear(tabla, RECETA),
)

print(f"Mercado del sancocho en {CIUDAD}, {MES}: ${costo_sancocho(df):,.0f}")

# ✏️ Dibujen el recorrido de `df` por las cuatro funciones.
# ¿Por qué una lambda permite fijar ciudad, mes y receta sin romper la pureza de las funciones?

🗣️ **Confronten P1:** ¿quién tuvo el mejor ojo de mercado? Los datos son precios mayoristas de una central de abastos: ¿el mercado de la esquina cobraría más o menos y por qué?

In [ ]:
# ¿Qué ingrediente manda? Confronten P2 con los aportes al costo.
mes_tabla = filtrar_productos(
    filtrar_mes(filtrar_ciudad(df, CIUDAD), MES), RECETA
)
precios_mes = mes_tabla.groupby("producto")["precio_kg"].mean()
aportes = pd.Series({
    producto: kilos * precios_mes[producto]
    for producto, kilos in RECETA.items()
}).sort_values()

ax = aportes.plot.barh(figsize=(8, 3.5), color="#B76E2E")
ax.set_xlabel(f"Aporte al costo del sancocho ($, {MES})")
ax.set_title("¿Qué ingrediente manda en la olla?")
plt.tight_layout()
plt.show()

total = aportes.sum()
espinazo = aportes["Carne de cerdo, espinazo"]
print(f"El espinazo aporta ${espinazo:,.0f} de ${total:,.0f}: {100 * espinazo / total:.0f}% de la olla.")

# ✏️ Para P3, cambien la receta a solo papa y plátano; comparen el primer y último mes.

---
## 4 · Momento 3 — Explicar

🗣️ Media página del equipo: confronten P1 y P2, expliquen cómo decidieron los nombres oficiales y completen la frase: **«Un pipeline es una composición de funciones porque…»**

---
## 🤖 Protocolo de IA

Las reglas de siempre siguen vigentes: intento propio, la IA no es autoridad final y todo uso va en la bitácora individual. Para la cacería estrella pregunten a una IA, sin contexto: *«¿cuánto cuesta hacer un sancocho para seis personas en Colombia?»*. Guarden la respuesta y la fecha. En la sesión 2 la compararán con el índice construido sobre datos oficiales.

**Próxima sesión:** el índice en el tiempo, los picos históricos, el duelo de ciudades, la conmutatividad y la unidad sancocho. Averigüen antes qué ingredientes lleva el plato de su casa.